# R2 모델 집단 학습 + 점수 저장

집단별 해결 확률의 차이로 쌍별 순위를 학습합니다. 동률 가중치는 기존 R2와 같은 0.25입니다.

기존 EV-10의 RouterBench·xRouteBench·LLMRouterBench·SPROUT 모델 집단과
`ours/data/blended`를 그대로 씁니다. 집단의 **해결 확률**이 학습 target이며,
과거 대표 모델의 연속 품질 점수와 의미가 다릅니다.
ModernBERT-base, 512토큰, 3 epoch, seed 42, 원천별 동일 총 가중치,
실효 batch 32, 비용 .2/.6/1.0을 기존 R0·R1과 맞춥니다.

**런타임: T4 GPU, Python 3.12.** 새 LLM API 호출은 없습니다.
입력은 기존 `run001`의 준비 데이터와 G 평가 산출물입니다.
결과는 `run001/r23-extension`에 저장됩니다.
학습은 기존 trainer와 같아 중간 재개 기능이 없습니다. 연결이 끊기면
완료 여부를 확인하고 미완료 로컬 학습만 새 폴더에서 다시 실행하세요.
기존 R0·R1과 논문 라우터는 다시 학습하지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
REPO = Path('/content/router-final-r23-source')
SOURCE_ZIP = RUN / 'code/router-final-source-r23.zip'
EXPECTED_SHA = 'd5dac9c40bac97ce05b7ca7e4cd07fcc4ffe9c703b2874c376ff9753dedee62e'
if not SOURCE_ZIP.is_file():
    from google.colab import files
    print('배포 묶음 안의 router-final-source-r23.zip을 선택하세요.')
    uploaded = files.upload()
    SOURCE_ZIP = Path('/content/router-final-source-r23.zip')
    SOURCE_ZIP.write_bytes(uploaded['router-final-source-r23.zip'])
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(SOURCE_ZIP) == EXPECTED_SHA, '노트북과 소스 ZIP의 버전이 다릅니다.'
with zipfile.ZipFile(SOURCE_ZIP) as bundle:
    for name in bundle.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    bundle.extractall(REPO)
for name, digest in json.loads((REPO / 'source-bundle-manifest.json').read_text()).items():
    assert sha(REPO / name) == digest, name
assert sys.version_info[:2] in {(3, 11), (3, 12)}, 'Python 3.12 런타임을 선택하세요.'


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision', 'torchaudio'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.5.1',
                '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                str(REPO / 'ml/notebooks/router_final/requirements-train-colab.txt')], check=True)


계산은 별도 Python 프로세스에서 실행하므로 설치 후 세션을 다시 시작할 필요가 없습니다. 아래 경로가 기존 실험의 Drive 폴더인지 확인하세요.


In [ ]:
from pathlib import Path
import os, subprocess, sys
SCRIPT = REPO / 'ml/notebooks/router_final/r23_extension.py'
ENV = dict(os.environ, PYTHONPATH=str(REPO / 'ml/src'), PYTHONUNBUFFERED='1',
           TOKENIZERS_PARALLELISM='false')
def run_logged(command, *args):
    cmd = [sys.executable, str(SCRIPT), command, '--run-dir', str(RUN), '--repo', str(REPO),
           *map(str, args)]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=ENV)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait():
        raise RuntimeError(f'{command} 실패 (exit {proc.returncode}). 위 오류를 확인하세요.')
run_logged('preflight')


In [ ]:
METHOD = 'r2_group'
LOCAL = Path('/content') / ('ev10-' + METHOD)
SAVED = RUN / 'r23-extension/models' / METHOD
subprocess.run([sys.executable, '-c',
    "import torch; assert torch.cuda.is_available(), 'T4 GPU를 선택하세요'; "
    "print(torch.cuda.get_device_name(0), torch.__version__)"], env=ENV, check=True)


## 1. 시험 학습

64행·2 step으로 loss와 런타임을 확인합니다. 시험 모델은 본 평가에 들어갈 수 없습니다.


In [ ]:
import tempfile
if not (SAVED / '_SUCCESS').exists():
    SMOKE = Path(tempfile.mkdtemp(prefix=METHOD + '-smoke-', dir='/content'))
    run_logged('train', '--method', METHOD, '--output-dir', SMOKE, '--device', 'cuda', '--smoke')


## 2. 본 학습과 Drive 보존

완료된 모델이 있으면 무결성을 확인하고 재사용합니다. 새 모델은 로컬에서 학습을 마친 뒤 Drive에 복사·재검증합니다.


In [ ]:
if (SAVED / '_SUCCESS').exists():
    run_logged('train', '--method', METHOD, '--output-dir', SAVED, '--device', 'cuda')
else:
    run_logged('train', '--method', METHOD, '--output-dir', LOCAL, '--device', 'cuda')
    run_logged('publish', '--method', METHOD, '--local-model', LOCAL)


## 3. dev·test 질문의 점수 저장

각 질문의 세 logit을 저장합니다. test에서는 질문 텍스트와 ID만 읽으며 성패 정답은 읽지 않습니다. 기존 07a와 같은 512토큰 추론입니다.


In [ ]:
run_logged('score', '--method', METHOD, '--device', 'cuda')


In [ ]:
import json
print(json.dumps(json.loads((SAVED / 'metrics.json').read_text()), indent=1)[:4000])
print('점수:', RUN / 'r23-extension/scores' / METHOD / 'scores.parquet')


R2와 R3의 점수 저장이 모두 끝나면 `13_r23_gain.ipynb`를 CPU 런타임에서 실행하세요.
